# Eda Income Insurance Ecommerce

EDA-проект по трем наборам данных:
1) доходы жителей Краснодара;
2) стоимость медицинской страховки;
3) поведение пользователей в e-commerce.

Идея — показать, как я работаю с исследовательским анализом данных на практике.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

## 1. Базовые настройки

Делаем вывод таблиц в консоли удобнее:
- видны все столбцы;
- ширина строки не обрезается;
- текст в ячейках не сокращается.

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## 2. Анализ доходов жителей Краснодара

Загружаем таблицу с доходами.

index_col=0: первый столбец используется как индекс, а не как обычный признак.

In [ ]:
income_df = pd.read_csv('data/krasnodar_income.csv', sep=',', header=0, index_col=0)

При необходимости можно посмотреть общую сводку по датасету.

In [ ]:
print(income_df.describe(include='all'))

Выделяем отдельную подгруппу — IT-специалистов.

In [ ]:
it_income_df = income_df[income_df['профессия'] == 'IT-специалист']

При необходимости можно посмотреть статистику только по IT-сегменту.

In [ ]:
print(it_income_df.describe())

In [ ]:
# Гистограмма доходов всех жителей.
plt.hist(income_df['доход_руб'], bins=11, density=True)

plt.title('Гистограмма доходов жителей Краснодара')
plt.xlabel('Доход, руб.')
plt.ylabel('Плотность')

plt.show()

In [ ]:
# Гистограмма доходов IT-специалистов.
plt.hist(it_income_df['доход_руб'], bins=69, density=True)

plt.title('Гистограмма доходов IT-специалистов в Краснодаре')
plt.xlabel('Доход, руб.')
plt.ylabel('Плотность')

plt.show()

In [ ]:
# Ядерная оценка плотности (KDE) для всех жителей.
# gaussian_kde позволяет сгладить "рваную" форму гистограммы
# и посмотреть на распределение более плавно.
income_kde = gaussian_kde(income_df['доход_руб'], bw_method=1)
income_grid = np.linspace(income_df['доход_руб'].min(), income_df['доход_руб'].max(), 1000)
plt.plot(income_grid, income_kde(income_grid), 'g', label='Все жители')

# Ядерная оценка плотности для IT-специалистов.
it_income_kde = gaussian_kde(it_income_df['доход_руб'], bw_method=1)
it_income_grid = np.linspace(it_income_df['доход_руб'].min(), it_income_df['доход_руб'].max(), 1000)
plt.plot(it_income_grid, it_income_kde(it_income_grid), 'b', label='IT-специалисты')

plt.title('Сравнение распределений доходов')
plt.xlabel('Доход, руб.')
plt.ylabel('Плотность')
plt.legend()
plt.show()

## 3. Анализ стоимости страховки

Идея блока:

проверить, как связаны стоимость страховки, факт курения и BMI.

In [ ]:
insurance_df = pd.read_csv('data/insurance.csv', header=0)

Разделяем клиентов на курящих и некурящих.

In [ ]:
smokers_df = insurance_df[insurance_df['smoker'] == 'yes']
non_smokers_df = insurance_df[insurance_df['smoker'] == 'no']

Дополнительная гипотеза:

двойной пик у курильщиков может объясняться тем, что в выборке смешались две группы:
1) курильщики с BMI в пределах нормы;
2) курильщики с BMI ниже нормы или выше нормы.

In [ ]:
smokers_bmi_outside_norm_df = insurance_df[
    (insurance_df['smoker'] == 'yes') &
    ((insurance_df['bmi'] > 35) | (insurance_df['bmi'] < 18.5))
]

In [ ]:
smokers_bmi_normal_df = insurance_df[
    (insurance_df['smoker'] == 'yes') &
    (18.5 <= insurance_df['bmi']) &
    (insurance_df['bmi'] <= 35)
]

In [ ]:
# KDE для стоимости страховки у курящих клиентов.
smokers_charges_kde = gaussian_kde(smokers_df['charges'], bw_method=0.1)
smokers_charges_grid = np.linspace(smokers_df['charges'].min(), smokers_df['charges'].max(), 1000)
plt.plot(smokers_charges_grid, smokers_charges_kde(smokers_charges_grid), 'g')

plt.title('Плотность распределения страховых выплат: курящие')
plt.xlabel('Стоимость страховки')
plt.ylabel('Плотность')
plt.show()

In [ ]:
# KDE для стоимости страховки у некурящих клиентов.
non_smokers_charges_kde = gaussian_kde(non_smokers_df['charges'], bw_method=0.1)
non_smokers_charges_grid = np.linspace(non_smokers_df['charges'].min(), non_smokers_df['charges'].max(), 1000)
plt.plot(non_smokers_charges_grid, non_smokers_charges_kde(non_smokers_charges_grid), 'b')

plt.title('Плотность распределения страховых выплат: некурящие')
plt.xlabel('Стоимость страховки')
plt.ylabel('Плотность')
plt.show()

In [ ]:
# Проверяем гипотезу о "двойном пике" среди курильщиков.
plt.hist(smokers_bmi_outside_norm_df['charges'])
plt.title('Курильщики с BMI вне нормы')
plt.show()

plt.hist(smokers_bmi_normal_df['charges'])
plt.title('Курильщики с BMI в пределах нормы')
plt.show()

## 4. Анализ поведения пользователей в e-commerce

Идея блока:

сравнить новые и возвращающиеся пользователи,
а также посмотреть, как сумма покупки может быть связана
с временем, проведенным на сайте, и типом устройства.

In [ ]:
ecommerce_df = pd.read_csv('data/ecommerce_user_behavior.csv', header=0, index_col=0)

Делим пользователей по типу клиента.

In [ ]:
new_customers_df = ecommerce_df[ecommerce_df['customer_type'] == 'new']
returning_customers_df = ecommerce_df[ecommerce_df['customer_type'] == 'returning']

In [ ]:
# Сравнение распределений суммы покупки у новых и постоянных клиентов.
plt.hist(new_customers_df['purchase_value'], bins=5, density=True)
plt.title('Стоимость покупки у новых клиентов')
plt.xlabel('Сумма покупки')
plt.ylabel('Плотность')
plt.show()

plt.hist(returning_customers_df['purchase_value'], bins=5, density=True)
plt.title('Стоимость покупки у возвращающихся клиентов')
plt.xlabel('Сумма покупки')
plt.ylabel('Плотность')
plt.show()

Разбиваем пользователей на группы по времени на сайте.

Такой разрез помогает посмотреть, меняется ли распределение purchase_value
вместе с ростом вовлеченности в сессию.

In [ ]:
users_short_session_df = ecommerce_df[ecommerce_df['time_spent_minutes'] <= 9.75]
users_medium_session_df = ecommerce_df[
    (ecommerce_df['time_spent_minutes'] > 9.75) &
    (ecommerce_df['time_spent_minutes'] <= 17.5)
]
users_long_session_df = ecommerce_df[
    (ecommerce_df['time_spent_minutes'] > 17.5) &
    (ecommerce_df['time_spent_minutes'] < 30.5)
]
users_very_long_session_df = ecommerce_df[ecommerce_df['time_spent_minutes'] >= 30.5]

In [ ]:
plt.hist(users_short_session_df['purchase_value'])
plt.title('Покупки пользователей со временем на сайте до 10 минут')
plt.show()

plt.hist(users_medium_session_df['purchase_value'])
plt.title('Покупки пользователей со временем на сайте 10–20 минут')
plt.show()

plt.hist(users_long_session_df['purchase_value'])
plt.title('Покупки пользователей со временем на сайте 20–30 минут')
plt.show()

plt.hist(users_very_long_session_df['purchase_value'])
plt.title('Покупки пользователей со временем на сайте более 30 минут')
plt.show()

In [ ]:
# Ниже — заготовка для регрессионного графика.
# Блок оставлен как идея для развития анализа.
import seaborn as sns
from scipy import stats

plt.figure(figsize=(12, 8))
sns.regplot(
    x='time_spent_minutes',
    y='purchase_value',
    data=ecommerce_df,
    scatter_kws={'alpha': 0.6, 's': 80},
    line_kws={'color': 'red', 'linewidth': 3}
)

plt.title('Зависимость суммы покупки от времени на сайте', fontsize=16, pad=20)
plt.xlabel('Время на сайте (минуты)', fontsize=12)
plt.ylabel('Сумма покупки', fontsize=12)
plt.grid(True, alpha=0.3)

correlation = ecommerce_df['time_spent_minutes'].corr(ecommerce_df['purchase_value'])
plt.text(
    5,
    ecommerce_df['purchase_value'].max() * 0.9,
    f'Коэффициент корреляции: {correlation:.2f}',
    fontsize=12,
    bbox=dict(facecolor='white', alpha=0.8)
)

plt.tight_layout()
plt.show()

slope, intercept, r_value, p_value, std_err = stats.linregress(
    ecommerce_df['time_spent_minutes'],
    ecommerce_df['purchase_value']
)
print(f"Уравнение регрессии: y = {slope:.2f}x + {intercept:.2f}")
print(f"R² = {r_value**2:.3f}")

Дополнительный разрез по типу устройства.

In [ ]:
mobile_users_df = ecommerce_df[ecommerce_df['device_type'] == 'mobile']
desktop_users_df = ecommerce_df[ecommerce_df['device_type'] == 'desktop']
tablet_users_df = ecommerce_df[ecommerce_df['device_type'] == 'tablet']

In [ ]:
print(mobile_users_df.describe(include='all'))
print(tablet_users_df.describe(include='all'))
print(desktop_users_df.describe(include='all'))

---
**Примечание:** для запуска ноутбука достаточно открыть его в корне соответствующего репозитория — все файлы данных лежат рядом и читаются по относительному пути.